# 🧱 Pydantic — Learning from First Principles

**Goal of this notebook:** understand *why* Pydantic exists, and then learn its core features step by step:
data validation, type coercion, custom validators, computed fields, nested models and serialization.

> Pydantic version used here: **v2** (`from pydantic import ...`). Some older tutorials show v1 syntax (`@validator`, `.dict()`, `.json()`), which is now replaced by `@field_validator`, `.model_dump()`, `.model_dump_json()`.

## 📑 Table of Contents
1. Why do we need Pydantic? (the problem)
2. Your first Pydantic model
3. Using a model inside functions
4. Complex types: `List`, `Dict`
5. Required vs Optional fields
6. Built-in special types: `EmailStr`, `AnyUrl`
7. `Field()` — constraints and metadata
8. Type coercion and `strict` mode
9. `@field_validator` — custom validation for one field
10. `@model_validator` — validation across multiple fields
11. `@computed_field` — derived fields
12. Nested models
13. Serialization — `model_dump()` and `model_dump_json()`
14. Summary cheat-sheet and practice exercises

### ⚙️ Setup
Run this once if `pydantic` or `email-validator` (needed for `EmailStr`) is not installed.

In [1]:
# Uncomment and run once if needed
# %pip install "pydantic[email]"

import pydantic
print("Pydantic version:", pydantic.VERSION)

Pydantic version: 2.13.5


---
# 1️⃣ Why do we need Pydantic?

Python is **dynamically typed**: it never checks whether the data passed to a function is of the "right" kind.
This is convenient, but dangerous when the data goes into a database, an API or an ML pipeline.

We will walk through the problem the way it happens in a real team, and see how each "fix" has a weakness. That weakness is exactly where Pydantic comes in.

## Problem 1 — Python does not enforce types

In [2]:
# 👨‍💼 Senior programmer writes this function
def insert_patient_data(name, age):
    print(name)
    print(age)
    print('inserted into database')

In [3]:
# 👶 Junior programmer calls it
insert_patient_data('raju', 23)              # ✅ works, as expected
insert_patient_data('raju', 'twenty three')  # ⚠️ ALSO works. A string was stored as an age!

raju
23
inserted into database
raju
twenty three
inserted into database


> ⚠️ **Problem:** bad data (`'twenty three'`) silently entered our "database". Later, `age + 1` or `age > 60` will crash or give wrong results.

## Solution attempt 1 — Type hinting

In [4]:
# 👨‍💼 Senior programmer adds type hints
def insert_patient_data(name: str, age: int):
    print(name)
    print(age)
    print('inserted into database')

In [5]:
# 👶 Junior programmer ignores the hint
insert_patient_data(name='raju', age='thirty')   # ⚠️ still runs!

raju
thirty
inserted into database


> ❌ **Why it fails:** type hints are only *documentation* for humans and tools (like your IDE or `mypy`).
> **Python does not enforce them at runtime.**

## Solution attempt 2 — Manual validation

In [6]:
def insert_patient_data(name: str, age: int):
    # Manual type check
    if type(name) == str and type(age) == int:
        print(name)
        print(age)
        print('inserted into database')
    else:
        raise TypeError('Incorrect data type')

In [7]:
try:
    insert_patient_data(name='raju', age='thirty')
except TypeError as e:
    print("❌ Caught:", e)

❌ Caught: Incorrect data type


✅ Now the wrong type is rejected. But two new problems appear:

1. **Type validation ≠ data validation.** What if `age = -5`? It is an `int`, so it passes, but it is clearly invalid.
   We would have to write *more* `if` statements (age must be > 0, name must not be empty, ...).
2. **Repetitive code.** Every function that accepts patient data (`insert_patient_data`, `update_patient_data`, ...)
   needs the same checks copy-pasted. It is error-prone and hard to maintain.

## ✅ The real solution — Pydantic
Python has no built-in validation, so **Pydantic** was created. It lets us

- describe the **shape of our data once** (a *schema*),
- get **type validation + data validation** automatically,
- get **type coercion** (safe conversion, e.g. `'23'` → `23`),
- reuse the same schema everywhere.

### The 3-step Pydantic workflow
| Step | What we do |
|------|------------|
| **1** | Define a Pydantic model (a class inheriting from `BaseModel`) |
| **2** | Create an object by passing raw data — validation happens automatically |
| **3** | Pass that *validated object* to your functions |

---
# 2️⃣ Your first Pydantic model

## Step 1 — Define the schema
A schema is a class that inherits from `BaseModel`. Each attribute is annotated with its type.

In [8]:
from pydantic import BaseModel

class Patient(BaseModel):
    name: str
    age: int

## Step 2 — Create an object from raw data
Notice that `age` is given as the **string** `'23'`. Pydantic converts it into the integer `23` (this is called *type coercion*, see section 8).

We use `**patient_info` to *unpack* the dictionary into keyword arguments: `Patient(name=..., age=...)`.

In [9]:
patient_info = {'name': 'raju joshi', 'age': '23'}

patient1 = Patient(**patient_info)   # validation happens here
patient1

Patient(name='raju joshi', age=23)

In [10]:
print(patient1.name, type(patient1.name))
print(patient1.age, type(patient1.age))   # <class 'int'>. Coerced from '23'

raju joshi <class 'str'>
23 <class 'int'>


### What if the data is invalid?
Pydantic raises a `ValidationError` with a clear, structured message showing **which field** failed and **why**.

In [11]:
from pydantic import ValidationError

try:
    Patient(name='raju', age='thirty')     # 'thirty' can't be converted into an int
except ValidationError as e:
    print(e)

1 validation error for Patient
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='thirty', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing


---
# 3️⃣ Step 3 — Use the validated object in functions
Instead of taking loose values (`name`, `age`), the function accepts **one validated `Patient` object**.
Inside the function we can trust that the data is already clean.

In [12]:
def insert_patient_data(patient: Patient):
    print(patient.name)
    print(patient.age)
    print('data inserted successfully')

insert_patient_data(patient1)

raju joshi
23
data inserted successfully


### 🔁 It's reusable / scalable
Another function can use the *same* schema. No validation code is repeated. This solves the "repetitive coding" problem.

In [13]:
def update_patient_data(patient: Patient):
    print(patient.name)
    print(patient.age)
    print('data updated successfully')

update_patient_data(patient1)

raju joshi
23
data updated successfully


---
# 4️⃣ Complex types: `List` and `Dict`

Real data has collections: a list of allergies, a dictionary of contact details, and so on.
We should also validate **what is inside** the collection (every allergy must be a `str`).

### `list[str]` (built-in generic)
In modern Pydantic v2 (and Python 3.9+) `list[str]` validates **both** levels:
1. the value is a list, and
2. each item inside is a string.

In [14]:
class Patient(BaseModel):
    name: str
    age: int
    allergies: list[str]

p1 = {'name': 'raju', 'age': 23, 'allergies': ['dust', 'smoke']}
patient1 = Patient(**p1)
patient1

Patient(name='raju', age=23, allergies=['dust', 'smoke'])

In [15]:
# An int (123) inside a list of strings → validation error
p2 = {'name': 'raju', 'age': 23, 'allergies': ['dust', 'smoke', 123]}

try:
    Patient(**p2)
except ValidationError as e:
    print(e)      # notice the location: allergies.2  → item at index 2

1 validation error for Patient
allergies.2
  Input should be a valid string [type=string_type, input_value=123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type


### `typing.List` vs built-in `list`
| Style | Example | Notes |
|-------|---------|-------|
| Old style | `from typing import List` → `List[str]` | Required before Python 3.9. Also works in v2. |
| Modern style | `list[str]`, `dict[str, str]` | Preferred on Python 3.9+ |

> 📝 **Class note:** in the *early days of Pydantic (v1)* the plain `list[str]` did not always give the "two-level" validation, so `typing.List[str]` was recommended.
> In **Pydantic v2 both forms behave the same** (as the cell above shows). We keep `List[str]` below because you will see it in many codebases.

In [16]:
from typing import List, Dict

class Patient(BaseModel):
    name: str
    age: int
    allergies: List[str]             # level 1: is it a list?  level 2: is each item a str?
    contact_details: Dict[str, str]  # keys must be str, values must be str

Patient(
    name='raju', age=23,
    allergies=['dust', 'smoke'],
    contact_details={'phone': '9876543210'}
)

Patient(name='raju', age=23, allergies=['dust', 'smoke'], contact_details={'phone': '9876543210'})

---
# 5️⃣ Required vs Optional fields

- **By default every field is required.** If it is missing, Pydantic raises an error.
- To make a field optional, wrap the type in `Optional[...]` **and give it a default value** (usually `None`).

In [17]:
class Patient(BaseModel):
    name: str
    age: int
    allergies: List[str]     # required (no default)

try:
    Patient(name='raju', allergies=['dust', 'smoke'])   # 'age' is missing
except ValidationError as e:
    print(e)                                            # → "Field required"

1 validation error for Patient
age
  Field required [type=missing, input_value={'name': 'raju', 'allergies': ['dust', 'smoke']}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


In [18]:
from typing import Optional

class Patient(BaseModel):
    name: str
    age: int
    allergies: Optional[List[str]] = None   # optional → MUST have a default

patient1 = Patient(name='raju', age=23)     # allergies not given → None
print(patient1)

name='raju' age=23 allergies=None


> ⚠️ **Important (Pydantic v2):** `Optional[X]` on its own only means *"may be `None`"*, it does **not** make the field optional to pass.
> You must also write `= None` (or another default). In v1 the default was implied, in v2 it isn't.
>
> Modern equivalent: `allergies: list[str] | None = None`

---
# 6️⃣ Built-in special types: `EmailStr`, `AnyUrl`

Pydantic ships with ready-made types that validate *formats* — no regex needed.

| Type | Validates |
|------|-----------|
| `EmailStr` | a valid email address (needs the `email-validator` package) |
| `AnyUrl` | a valid URL |
| others | `HttpUrl`, `IPvAnyAddress`, `UUID4`, `PositiveInt`, `conint`, ... |

In [19]:
from pydantic import EmailStr, AnyUrl

class Patient(BaseModel):
    name: str
    age: int
    allergies: Optional[List[str]] = None
    email: EmailStr
    github: AnyUrl

patient1 = Patient(
    name='raju joshi',
    age=23,
    email='rajujoshi@gmail.com',
    github='https://github.com/Rajendra-Pd-Joshi',
)
patient1

Patient(name='raju joshi', age=23, allergies=None, email='rajujoshi@gmail.com', github=AnyUrl('https://github.com/Rajendra-Pd-Joshi'))

In [20]:
# Invalid e-mail → rejected
try:
    Patient(
        name='raju joshi',
        age=23,
        email='rajujoshi.com',      # no '@' → invalid
        github='https://github.com/Rajendra-Pd-Joshi',
    )
except ValidationError as e:
    print(e)

1 validation error for Patient
email
  value is not a valid email address: An email address must have an @-sign. [type=value_error, input_value='rajujoshi.com', input_type=str]


---
# 7️⃣ `Field()` — custom constraints and metadata

Built-in types cover common formats, but **business rules** are specific to *your* application
(e.g. "weight must be between 0 and 120 kg", "name can be at most 20 characters").

`Field()` lets us attach two kinds of information to a field:
1. **Constraints** (data validation)
2. **Metadata** (documentation)

### Common `Field` constraints
| Argument | Meaning | Applies to |
|----------|---------|-----------|
| `gt`, `ge` | greater than, greater than or equal | numbers |
| `lt`, `le` | less than, less than or equal | numbers |
| `min_length`, `max_length` | length limits | `str`, `list`, `dict` |
| `pattern` | regex the string must match | `str` |
| `default` | default value | any |
| `strict` | disable type coercion for this field | any |
| `description`, `title`, `examples` | metadata | any |

In [21]:
from pydantic import Field

class Person(BaseModel):
    weight: float = Field(gt=0, le=120)                     # 0 < weight <= 120
    name: str = Field(max_length=20)                        # at most 20 characters
    allergies: Optional[List[str]] = Field(default=None, max_length=20)  # list can hold at most 20 items

def display_info(p: Person):
    print(p.name)
    print(p.weight)
    print(p.allergies)

In [22]:
person1 = Person(weight=63, name='raju')
display_info(person1)

raju
63.0
None


In [23]:
# Violating the constraints
try:
    Person(weight=150, name='raju')          # weight > 120
except ValidationError as e:
    print(e)

1 validation error for Person
weight
  Input should be less than or equal to 120 [type=less_than_equal, input_value=150, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal


> 💡 Note: `max_length` on a `List` limits the **number of items**, not the length of each string.

## `Field` also stores metadata
Metadata does not validate anything; it documents the field. It shows up in the auto-generated **JSON schema** and in API docs (e.g. FastAPI's Swagger UI).

### The `Annotated` style
`Annotated[type, Field(...)]` keeps the type and its rules together. It is the recommended style in modern Pydantic.

In [24]:
from typing import Annotated

class Person(BaseModel):
    weight: Annotated[float, Field(gt=0, le=120, description='Weight of the person in kg')]
    name: Annotated[str, Field(max_length=20, title='Person name',
                               description='Give the name of the patient',
                               examples=['raju'])]
    allergies: Annotated[Optional[List[str]], Field(default=None, max_length=20)]

Person(weight=63, name='raju')

Person(weight=63.0, name='raju', allergies=None)

In [25]:
# The metadata becomes part of the JSON schema
import json
print(json.dumps(Person.model_json_schema(), indent=2))

{
  "properties": {
    "weight": {
      "description": "Weight of the person in kg",
      "exclusiveMinimum": 0,
      "maximum": 120,
      "title": "Weight",
      "type": "number"
    },
    "name": {
      "description": "Give the name of the patient",
      "examples": [
        "raju"
      ],
      "maxLength": 20,
      "title": "Person name",
      "type": "string"
    },
    "allergies": {
      "anyOf": [
        {
          "items": {
            "type": "string"
          },
          "maxItems": 20,
          "type": "array"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "title": "Allergies"
    }
  },
  "required": [
    "weight",
    "name"
  ],
  "title": "Person",
  "type": "object"
}


### ⚠️ Correction of a mistake from the live class
In the live-class code the `name` field was written as `Annotated[int, Field(max_length=20, ...)]`.
That is a **bug**: `name` should be `str` (and `max_length` makes no sense on an `int`). It is fixed above.

### ⚠️ Deprecated: extra keyword arguments on `Field`
Older code writes `Field(..., example='raju')`. Passing **unknown / extra keyword arguments** like `example=` is **deprecated in Pydantic v2**.
Use `examples=[...]` (a list) or `json_schema_extra={...}` instead.

In [26]:
# ❌ Deprecated way (emits a warning)
# name: str = Field(max_length=20, example='raju')

# ✅ Recommended way
class Person(BaseModel):
    weight: float = Field(gt=0, le=120, description='Person weight in kg')
    name: str = Field(max_length=20, title='Name of a person',
                      description='Person name', examples=['raju'])
    allergies: Optional[List[str]] = Field(default=None, max_length=20)

---
# 8️⃣ Type coercion (and how to turn it off)

**Type coercion** = Pydantic tries to *convert* the input into the declared type before giving up.

| Input | Declared | Result |
|-------|----------|--------|
| `'12'` | `int` | `12` ✅ |
| `'34.5'` | `float` | `34.5` ✅ |
| `'abc'` | `int` | ❌ error |

This is convenient (data from forms, JSON, CSV often arrives as strings) but sometimes **too permissive**.

In [27]:
class Person(BaseModel):
    name: str
    age: int
    weight: float

p1 = Person(name='raju', age='12', weight='34.5')   # no error → strings are coerced
print(p1)
print(type(p1.age), type(p1.weight))

name='raju' age=12 weight=34.5
<class 'int'> <class 'float'>


### Strict mode
In production you may *not* want silent conversion. Use `Field(strict=True)` and Pydantic will accept **only the exact type**.

In [28]:
class Person(BaseModel):
    name: str
    age: int = Field(strict=True)          # no coercion allowed
    weight: float = Field(strict=True)     # no coercion allowed

try:
    Person(name='raju', age='12', weight='34.5')
except ValidationError as e:
    print(e)

2 validation errors for Person
age
  Input should be a valid integer [type=int_type, input_value='12', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_type
weight
  Input should be a valid number [type=float_type, input_value='34.5', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/float_type


> 📝 You can also enable strict mode for the *whole model*: `model_config = ConfigDict(strict=True)`.

---
# 9️⃣ `@field_validator` — custom validation for a single field

`Field()` covers simple rules, but sometimes we need **custom logic**.
A field validator is a function that runs on **one field** and can:
- **validate** the value (raise `ValueError` if bad), and
- **transform** the value (return a modified value).

### Syntax
```python
@field_validator('field_name')
@classmethod
def my_validator(cls, value):
    ...            # check / transform
    return value   # ⚠️ ALWAYS return the value
```

### Exercise 1
> Only patients who are **HDFC or ICICI employees** get a 20 % discount. Validate the email domain of the patient.

In [29]:
from pydantic import field_validator

class Patient(BaseModel):
    email: EmailStr

    @field_validator('email')
    @classmethod
    def email_validator(cls, value):
        valid_domains = ['hdfc.com', 'icici.com']
        domain_name = value.split('@')[-1]          # 'abc@gmail.com' → 'gmail.com'

        if domain_name not in valid_domains:
            raise ValueError('Not a valid HDFC or ICICI email')
        return value

In [30]:
try:
    Patient(email='raju@gmail.com')            # ❌ wrong company
except ValidationError as e:
    print(e)

1 validation error for Patient
email
  Value error, Not a valid HDFC or ICICI email [type=value_error, input_value='raju@gmail.com', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


In [31]:
Patient(email='rajujoshi@hdfc.com')            # ✅ valid

Patient(email='rajujoshi@hdfc.com')

### Transforming a value
Because the validator *returns* the value, we can also clean it. Here we normalise the name to Title Case.

In [32]:
class Patient(BaseModel):
    name: str

    @field_validator('name')
    @classmethod
    def name_transform(cls, value):
        return value.strip().title()     # ' raju joshi ' → 'Raju Joshi'

Patient(name='  raju joshi ')

Patient(name='Raju Joshi')

## Modes of a field validator: `before` vs `after`

Pydantic processes a field in this order:

```
raw input  ──►  [ mode='before' validators ]  ──►  type coercion / validation  ──►  [ mode='after' validators ]  ──►  final value
```

| Mode | The validator receives | Typical use |
|------|-----------------------|-------------|
| `mode='after'` *(default)* | the value **after** it has been coerced to the declared type | business rules on clean data |
| `mode='before'` | the **raw** input exactly as the user passed it | pre-processing, or rejecting things Pydantic would auto-convert |

Let's see the difference with the same input `age='23'`.

In [33]:
# Baseline: no custom validator → '23' is coerced to 23
class Person(BaseModel):
    name: str
    age: int

Person(name='raju', age='23')

Person(name='raju', age=23)

### `mode='before'` → validator sees the raw value `'23'` (a `str`)

In [34]:
class Person(BaseModel):
    name: str
    age: int

    @field_validator('age', mode='before')
    @classmethod
    def age_validator(cls, value):
        # value is the RAW input (before coercion)
        if not isinstance(value, int):
            raise ValueError('age should be an integer')
        return value

try:
    Person(name='raju', age='23')          # '23' is a str at this point → error
except ValidationError as e:
    print(e)

1 validation error for Person
age
  Value error, age should be an integer [type=value_error, input_value='23', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


### `mode='after'` → validator sees the coerced value `23` (an `int`)

In [35]:
class Person(BaseModel):
    name: str
    age: int

    @field_validator('age', mode='after')
    @classmethod
    def age_validator(cls, value):
        # value is already an int here (coerced from '23')
        if value < 0:
            raise ValueError('age cannot be negative')
        return value

print(Person(name='raju', age='23'))       # ✅ works: '23' → 23 → then our check runs

name='raju' age=23


> 💡 **Rule of thumb:** use `after` (the default) for business rules, since you can rely on the type. Use `before` only when you need to see or reshape the raw input.

---
# 🔟 `@model_validator` — validation across multiple fields

### Problem statement
> If a patient is **older than 60**, their `contact_details` must contain an **emergency** number.

This rule depends on **two fields together** (`age` and `contact_details`).
A `field_validator` only sees one field (and other fields may not be validated yet), so it is the wrong tool.

✅ **Solution:** `@model_validator(mode='after')`, which runs **after all fields are validated**, and gets the whole model (`self`).

```python
@model_validator(mode='after')
def my_check(self):
    ...          # use self.age, self.contact_details, ...
    return self  # ⚠️ ALWAYS return the model
```

In [36]:
from pydantic import model_validator

class Patient(BaseModel):
    name: str
    email: EmailStr
    age: int
    weight: float
    married: bool
    allergies: List[str]
    contact_details: Dict[str, str]

    @model_validator(mode='after')
    def validate_emergency_contact(self):
        # Cross-field rule: age > 60  →  'emergency' contact is mandatory
        if self.age > 60 and 'emergency' not in self.contact_details:
            raise ValueError('Patients older than 60 must have an emergency contact')
        return self

In [37]:
def update_patient_data(patient: Patient):
    print(patient.name)
    print(patient.age)
    print(patient.allergies)
    print(patient.married)
    print('updated')

In [38]:
# ✅ Age 65 WITH an emergency contact → passes
patient_info = {
    'name': 'nitish', 'email': 'abc@icici.com', 'age': '65', 'weight': 75.2,
    'married': True, 'allergies': ['pollen', 'dust'],
    'contact_details': {'phone': '2353462', 'emergency': '235236'},
}

patient1 = Patient(**patient_info)
update_patient_data(patient1)

nitish
65
['pollen', 'dust']
True
updated


In [39]:
# ❌ Age 65 WITHOUT an emergency contact → fails
bad_info = {**patient_info, 'contact_details': {'phone': '2353462'}}

try:
    Patient(**bad_info)
except ValidationError as e:
    print(e)

1 validation error for Patient
  Value error, Patients older than 60 must have an emergency contact [type=value_error, input_value={'name': 'nitish', 'email...': {'phone': '2353462'}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


### `field_validator` vs `model_validator`
| | `field_validator` | `model_validator` |
|---|---|---|
| Scope | one field | whole model (many fields) |
| First argument | `cls` (with `@classmethod`) | `self` (mode `after`) |
| Use for | format / range / transform of a single value | rules that depend on several fields |

---
# 1️⃣1️⃣ `@computed_field` — derived fields

Some values can be **calculated** from other fields, so we should not ask the user to enter them.
Example: **BMI** = weight (kg) / height (m)². We already have `weight` and `height`.

`@computed_field` (stacked on top of `@property`) makes the calculated value behave like a real field.
It is included in `model_dump()` and in the JSON output.

In [40]:
from pydantic import computed_field

class Patient(BaseModel):
    name: str
    email: EmailStr
    age: int
    weight: float          # kg
    height: float          # metre
    married: bool
    allergies: List[str]
    contact_details: Dict[str, str]

    @computed_field
    @property
    def bmi(self) -> float:
        # BMI = weight / height²  (rounded to 2 decimals)
        return round(self.weight / (self.height ** 2), 2)

In [41]:
def update_patient_data(patient: Patient):
    print(patient.name)
    print(patient.age)
    print(patient.allergies)
    print(patient.married)
    print('BMI:', patient.bmi)
    print('updated')

patient_info = {
    'name': 'nitish', 'email': 'abc@icici.com', 'age': '65', 'weight': 75.2, 'height': 1.72,
    'married': True, 'allergies': ['pollen', 'dust'],
    'contact_details': {'phone': '2353462', 'emergency': '235236'},
}

patient1 = Patient(**patient_info)
update_patient_data(patient1)

nitish
65
['pollen', 'dust']
True
BMI: 25.42
updated


In [42]:
# The computed field is part of the serialized output too
patient1.model_dump()

{'name': 'nitish',
 'email': 'abc@icici.com',
 'age': 65,
 'weight': 75.2,
 'height': 1.72,
 'married': True,
 'allergies': ['pollen', 'dust'],
 'contact_details': {'phone': '2353462', 'emergency': '235236'},
 'bmi': 25.42}

> 📝 In the live class the property was named `calculate_bmi`. A **noun** (`bmi`) reads better for something that behaves like a field, so it is renamed here.

---
# 1️⃣2️⃣ Nested models

Real data is hierarchical: a patient **has an** address, an address has a city, state and pin.
Instead of one huge flat model, we create **smaller models and compose them**.

### Benefits
- ✅ **Reusability:** `Address` can be used by `Patient`, `Doctor`, `Hospital`, ...
- ✅ **Readability:** each model has one clear job.
- ✅ **Validation at every level:** the inner model is validated too.

In [43]:
class Address(BaseModel):
    city: str
    state: str
    pin: str

class Patient(BaseModel):
    name: str
    gender: str
    age: int
    address: Address       # a model used as a field type

In [44]:
# Way 1: build the inner object first, then pass it in
address_dict = {'city': 'gurgaon', 'state': 'haryana', 'pin': '122001'}
address1 = Address(**address_dict)

patient_dict = {'name': 'nitish', 'gender': 'male', 'age': 35, 'address': address1}
patient1 = Patient(**patient_dict)

patient1

Patient(name='nitish', gender='male', age=35, address=Address(city='gurgaon', state='haryana', pin='122001'))

In [45]:
# Way 2: pass a plain dict. Pydantic builds the nested Address automatically
patient2 = Patient(
    name='nitish', gender='male', age=35,
    address={'city': 'gurgaon', 'state': 'haryana', 'pin': '122001'},
)
print(patient2 == patient1)    # both are equal

True


In [46]:
# Access nested data with the dot notation
print(patient1.address)
print(patient1.address.city)
print(patient1.address.pin)

city='gurgaon' state='haryana' pin='122001'
gurgaon
122001


In [47]:
# Nested data is validated too: 'pin' must be a str, city is missing → errors
try:
    Patient(name='x', gender='male', age=1, address={'state': 'haryana', 'pin': 122001})
except ValidationError as e:
    print(e)      # locations look like address.city  /  address.pin

2 validation errors for Patient
address.city
  Field required [type=missing, input_value={'state': 'haryana', 'pin': 122001}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
address.pin
  Input should be a valid string [type=string_type, input_value=122001, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type


---
# 1️⃣3️⃣ Serialization — exporting a model

**Serialization** = converting a Pydantic object into a format we can store or send (dict, JSON string, ...).
This is what you do before saving to a database, writing to a file or returning an API response.

| Method | Returns | Notes |
|--------|---------|-------|
| `model_dump()` | Python `dict` | nested models become nested dicts |
| `model_dump_json()` | JSON `str` | ready to send over the network |

> 🔄 Old v1 names: `.dict()` → `.model_dump()`, `.json()` → `.model_dump_json()`

In [48]:
temp = patient1.model_dump()
print(temp)
print(type(temp))          # <class 'dict'>

{'name': 'nitish', 'gender': 'male', 'age': 35, 'address': {'city': 'gurgaon', 'state': 'haryana', 'pin': '122001'}}
<class 'dict'>


In [49]:
temp_json = patient1.model_dump_json()
print(temp_json)
print(type(temp_json))     # <class 'str'>

{"name":"nitish","gender":"male","age":35,"address":{"city":"gurgaon","state":"haryana","pin":"122001"}}
<class 'str'>


## Controlling the output
`model_dump()` accepts parameters to include / exclude data:

| Parameter | Effect |
|-----------|--------|
| `include={...}` | keep only these fields |
| `exclude={...}` | drop these fields |
| `exclude_unset=True` | drop fields the user **did not explicitly set** |
| `exclude_defaults=True` | drop fields whose value equals the default |
| `exclude_none=True` | drop fields whose value is `None` |

In [50]:
class Address(BaseModel):
    city: str
    state: str
    pin: str

class Patient(BaseModel):
    name: str
    gender: str = 'Male'          # has a default value
    age: int
    address: Address

address1 = Address(city='gurgaon', state='haryana', pin='122001')
patient1 = Patient(name='nitish', age=35, address=address1)   # 'gender' NOT provided

In [51]:
# exclude_unset=True → 'gender' was never set by us, so it is left out
temp = patient1.model_dump(exclude_unset=True)
print(temp)
print(type(temp))

{'name': 'nitish', 'age': 35, 'address': {'city': 'gurgaon', 'state': 'haryana', 'pin': '122001'}}
<class 'dict'>


In [52]:
# For comparison: everything is included by default (gender='Male' appears)
print(patient1.model_dump())

{'name': 'nitish', 'gender': 'Male', 'age': 35, 'address': {'city': 'gurgaon', 'state': 'haryana', 'pin': '122001'}}


In [53]:
# include / exclude: pick specific fields (works on nested fields too)
print(patient1.model_dump(include={'name', 'age'}))
print(patient1.model_dump(exclude={'address': {'pin'}}))     # hide only address.pin

{'name': 'nitish', 'age': 35}
{'name': 'nitish', 'gender': 'Male', 'age': 35, 'address': {'city': 'gurgaon', 'state': 'haryana'}}


> 💡 `exclude_unset=True` is very useful for **PATCH / partial-update** APIs: you only want to update the fields the client actually sent.

---
# 1️⃣4️⃣ Summary

## 🧠 Cheat-sheet
| Concept | Tool | One-line idea |
|---------|------|---------------|
| Schema | `class X(BaseModel)` | define the shape of the data once |
| Type validation | type hints (`int`, `str`, `List[str]`, `Dict[str,str]`) | Pydantic enforces the hints at runtime |
| Optional field | `Optional[T] = None` | must add a default |
| Format types | `EmailStr`, `AnyUrl` | ready-made validators |
| Constraints & metadata | `Field(gt=, le=, max_length=, description=)` | business rules + docs |
| Combined style | `Annotated[T, Field(...)]` | type and rules together |
| Type coercion | automatic (`'23'` → `23`) | turn off with `Field(strict=True)` |
| Single-field custom rule | `@field_validator` (`before` / `after`) | validate / transform one field |
| Cross-field rule | `@model_validator(mode='after')` | rules involving several fields |
| Derived value | `@computed_field` + `@property` | calculated, not entered |
| Composition | nested models | reuse + validation at each level |
| Export | `model_dump()`, `model_dump_json()` | dict / JSON, with include / exclude options |

## ⚠️ Common mistakes
1. Forgetting `= None` on an `Optional` field (it is still *required*).
2. Forgetting to **`return value`** inside a validator (the field becomes `None`).
3. Using `@field_validator` for rules that need **two fields** (use `@model_validator`).
4. Forgetting `@classmethod` under `@field_validator`.
5. Using v1 names (`@validator`, `.dict()`, `.json()`, `example=` in `Field`).
6. Expecting `Field(max_length=...)` on a `List` to limit *string length* (it limits the *number of items*).

## 🎯 Practice exercises
1. Create a `Doctor` model with `name`, `specialization`, `experience_years` (0-60) and a `hospital_email`.
2. Add a `field_validator` that rejects doctor names that do not start with `"Dr."`.
3. Add a `model_validator`: if `experience_years < 2`, `specialization` must not be `"Senior Consultant"`.
4. Add a `computed_field` `is_senior` that is `True` when `experience_years >= 10`.
5. Nest an `Address` inside `Doctor`, then export with `model_dump_json(exclude={'address': {'pin'}})`.
6. Extend Exercise 1 of section 9: add a `computed_field` called `discount_percent` that returns `20` for HDFC/ICICI emails.

## 🔗 References
- Official docs: <https://docs.pydantic.dev/latest/>
- Migration guide (v1 → v2): <https://docs.pydantic.dev/latest/migration/>